# 비염 환자 유형 정의서 분석 노트북

이 노트북은 기존 rhinitis_patient_profile.py 스크립트의 전체 분석 과정을 단계별로 셀로 분리하여, 데이터 로드, 전처리(결측치/중복/이상치 처리), 동반질환 클러스터링, 증상 텍스트 분석, 데이터 통합, 유형 정의서 생성 및 저장까지 포함합니다.

각 단계별로 데이터 수 변화도 함께 출력합니다.

## 1. 필요한 라이브러리 임포트
분석에 필요한 외부 및 표준 라이브러리를 import 합니다.

In [23]:
import os
import pandas as pd
# kagglehub이 설치되어 있지 않으면 pip로 설치 필요
# !pip install kagglehub
from kagglehub import dataset_download

## 2. 데이터 로드 및 준비
Childhood Allergies 데이터와 Healthcare.csv를 불러옵니다.

In [24]:
# Kaggle 데이터셋 다운로드 및 로드
path = dataset_download('thedevastator/childhood-allergies-prevalence-diagnosis-and-tre')
filepath = os.path.join(path, 'food-allergy-analysis-Zenodo.csv')
df = pd.read_csv(filepath)
print(f'원본 allergy 데이터 shape: {df.shape}')

# Healthcare.csv 로드
healthcare_path = os.path.join('..', 'data', 'raw', 'Healthcare.csv')
health = pd.read_csv(healthcare_path)
print(f'원본 healthcare 데이터 shape: {health.shape}')

원본 allergy 데이터 shape: (333200, 50)
원본 healthcare 데이터 shape: (25000, 6)


In [25]:
# AGE, GENDER 컬럼 생성 (컬럼명이 다를 경우 변환)
if 'Age' in df.columns and 'Gender' in df.columns:
    df['AGE'] = df['Age']
    df['GENDER'] = df['Gender']
    print('AGE, GENDER 컬럼 생성 완료')
else:
    print('원본 데이터에 Age, Gender 컬럼이 없습니다. 컬럼명을 확인하세요.')

원본 데이터에 Age, Gender 컬럼이 없습니다. 컬럼명을 확인하세요.


## 3. 데이터 전처리: 결측치, 중복, 이상치 처리
비염 환자 필터, 결측치/중복/이상치 처리 과정을 단계별로 수행하고 데이터 수 변화를 출력합니다.

In [26]:
print(df.columns)

Index(['SUBJECT_ID', 'BIRTH_YEAR', 'GENDER_FACTOR', 'RACE_FACTOR',
       'ETHNICITY_FACTOR', 'PAYER_FACTOR', 'ATOPIC_MARCH_COHORT',
       'AGE_START_YEARS', 'AGE_END_YEARS', 'SHELLFISH_ALG_START',
       'SHELLFISH_ALG_END', 'FISH_ALG_START', 'FISH_ALG_END', 'MILK_ALG_START',
       'MILK_ALG_END', 'SOY_ALG_START', 'SOY_ALG_END', 'EGG_ALG_START',
       'EGG_ALG_END', 'WHEAT_ALG_START', 'WHEAT_ALG_END', 'PEANUT_ALG_START',
       'PEANUT_ALG_END', 'SESAME_ALG_START', 'SESAME_ALG_END',
       'TREENUT_ALG_START', 'TREENUT_ALG_END', 'WALNUT_ALG_START',
       'WALNUT_ALG_END', 'PECAN_ALG_START', 'PECAN_ALG_END',
       'PISTACH_ALG_START', 'PISTACH_ALG_END', 'ALMOND_ALG_START',
       'ALMOND_ALG_END', 'BRAZIL_ALG_START', 'BRAZIL_ALG_END',
       'HAZELNUT_ALG_START', 'HAZELNUT_ALG_END', 'CASHEW_ALG_START',
       'CASHEW_ALG_END', 'ATOPIC_DERM_START', 'ATOPIC_DERM_END',
       'ALLERGIC_RHINITIS_START', 'ALLERGIC_RHINITIS_END', 'ASTHMA_START',
       'ASTHMA_END', 'FIRST_ASTHMARX', 'L

In [27]:
# 비염 환자만 필터 (동반질환 기반 클러스터링 준비)
rhinitis = df[df['ALLERGIC_RHINITIS_START'].notna()].copy()
print(f'비염 환자 필터 후: {len(rhinitis)}')

# rhinitis에 AGE, GENDER 컬럼 생성 (원본 컬럼명이 있을 때 변환)
if 'Age' in rhinitis.columns and 'Gender' in rhinitis.columns:
    rhinitis['AGE'] = rhinitis['Age']
    rhinitis['GENDER'] = rhinitis['Gender']
    print('rhinitis에 AGE, GENDER 컬럼 생성 완료')
else:
    print('rhinitis에 Age, Gender 컬럼이 없습니다. 컬럼명을 확인하세요.')

# 결측치 처리 (AGE, GENDER, ASTHMA_START, ATOPIC_DERM_START)
rhinitis_clean = rhinitis.dropna(subset=['AGE', 'GENDER'])
print(f'결측치 처리 후: {len(rhinitis_clean)}')

# 중복 환자 제거 (AGE+GENDER 기준 예시)
rhinitis_nodup = rhinitis_clean.drop_duplicates(subset=['AGE', 'GENDER'])
print(f'중복 제거 후: {len(rhinitis_nodup)}')

# 이상치 처리 (AGE 0~120세만)
rhinitis_final = rhinitis_nodup[(rhinitis_nodup['AGE'] > 0) & (rhinitis_nodup['AGE'] < 120)]
print(f'이상치 처리 후: {len(rhinitis_final)}')

비염 환자 필터 후: 55567
rhinitis에 Age, Gender 컬럼이 없습니다. 컬럼명을 확인하세요.


KeyError: ['AGE', 'GENDER']

In [21]:
# rhinitis에 AGE, GENDER 컬럼 생성 (원본 컬럼명이 있을 때 변환)
if 'Age' in rhinitis.columns and 'Gender' in rhinitis.columns:
    rhinitis['AGE'] = rhinitis['Age']
    rhinitis['GENDER'] = rhinitis['Gender']
    print('rhinitis에 AGE, GENDER 컬럼 생성 완료')
else:
    print('rhinitis에 Age, Gender 컬럼이 없습니다. 컬럼명을 확인하세요.')

rhinitis에 Age, Gender 컬럼이 없습니다. 컬럼명을 확인하세요.


## 4. 동반질환 클러스터링
천식, 아토피 동반 여부로 비염 환자 클러스터를 정의합니다.

In [ ]:
# 동반질환 클러스터링
def cluster_type(row):
    if pd.notna(row['ASTHMA_START']):
        return '비염+천식형'
    elif pd.notna(row['ATOPIC_DERM_START']):
        return '비염+아토피형'
    else:
        return '비염 단독형'
rhinitis_final['cluster'] = rhinitis_final.apply(cluster_type, axis=1)
print(rhinitis_final['cluster'].value_counts())

## 5. Healthcare 증상 텍스트 분석
증상 텍스트에서 주요 증상 키워드를 추출합니다.

In [ ]:
# 증상 키워드 매핑
symptom_keywords = {
    '콧물': ['runny nose'],
    '재채기': ['sneezing'],
    '코막힘': ['nasal congestion', 'congestion'],
    '눈가려움': ['eye itching', 'itchy eyes'],
}
def extract_symptoms(text):
    found = []
    for k, vlist in symptom_keywords.items():
        for v in vlist:
            if v in str(text).lower():
                found.append(k)
    return ','.join(found) if found else '기타'
health['증상_프로파일'] = health['Symptoms'].apply(extract_symptoms)
health['증상_프로파일'].value_counts().head()

## 6. 데이터 통합 및 유형 정의서 생성
비염 환자와 증상 프로파일을 통합하여 유형 정의서를 생성합니다.

In [ ]:
# 나이+성별 기준으로 데이터 통합 (실제 환경에서는 환자 ID 등 고유키 사용 권장)
merged = pd.merge(
    rhinitis_final,
    health,
    left_on=['AGE', 'GENDER'],
    right_on=['Age', 'Gender'],
    how='inner',
)
print(f'통합 데이터 shape: {merged.shape}')

# 유형 정의서 생성
profile = merged.groupby(['cluster', '증상_프로파일']).size().reset_index(name='환자수')
profile = profile.sort_values(['cluster', '환자수'], ascending=[True, False])
profile.head()

## 7. 결과 저장 및 시각화
유형 정의서를 csv로 저장하고, 표로 출력합니다.

In [ ]:
# 결과 저장
profile.to_csv('../outputs/reports/rhinitis_patient_profiles.csv', index=False, encoding='utf-8-sig')
print('비염 환자 유형 정의서 저장 완료: ../outputs/reports/rhinitis_patient_profiles.csv')

# 표로 출력
import IPython.display as disp
disp.display(profile)